# פירוק ספקטרלי של ספקטרוסקופיית רמאן באמצעות למידה עמוקה

## Raman Spectral Decomposition via Deep Learning

**פרויקט תזה** — אוניברסיטת בן-גוריון בנגב, המחלקה להנדסת תעשייה וניהול

---

### מה עושה המחברת הזו?

מחברת זו מציגה את **כל הפלטים והתוצאות** של הפרויקט בצורה מסודרת, עם הסברים בעברית.  
**לא צריך GPU או קלאסטר** — המחברת רק טוענת ומציגה תמונות שכבר נוצרו.

### מבנה:
1. חקירת נתונים
2. יצירת תערובות סינתטיות
3. בייסליין קלאסי (NNLS)
4. בדיקת Overfit
5. אימון מלא
6. הערכה והשוואה: DL v3 מול NNLS
7. גרפים מקצועיים ודוגמאות
8. טסט על נתונים אמיתיים (תערובות סוכר)

In [ ]:
# Setup: load images from outputs/figs/
from pathlib import Path
from IPython.display import display, Image, Markdown

# Auto-detect project root
# Works whether you run from notebooks/ or from deep2/
_nb_dir = Path('.').resolve()
if _nb_dir.name == 'notebooks':
    ROOT = _nb_dir.parent
else:
    ROOT = _nb_dir

FIGS = ROOT / 'outputs' / 'figs'

def show(path, width=900):
    """Display a PNG image from the figs directory."""
    full = FIGS / path
    if full.exists():
        display(Image(filename=str(full), width=width))
    else:
        print(f'Image not found: {full}')

print(f'Project root: {ROOT}')
print(f'Figures dir:  {FIGS}')
print(f'Found {len(list(FIGS.rglob("*.png")))} PNG files')

---

# שלב 1: חקירת נתונים

## Data Exploration

אספנו **115 ספקטרומים טהורים** של חומרים אורגניים ממספר מאגרים פתוחים:

| מאגר | תיאור | מספר ספקטרומים |
|-------|--------|---------------|
| RamanBioLib | ספריית ספקטרומים ביולוגיים/אורגניים | רוב הספקטרומים |
| SDBS (AIST) | מאגר מולקולות אורגניות | תוספת |
| Olive Oil | ספקטרומים ספציפיים למזון | מעט |
| Sugar Mixtures | תערובות סוכר מבוקרות | נתוני טסט |

כל הספקטרומים עברו:
- **אינטרפולציה** לגריד אחיד: 400–3400 cm⁻¹, רזולוציה 1 cm⁻¹ (3001 נקודות)
- **נורמליזציה L2** — כל הספקטרומים באותו סקאלה

### כיסוי ספקטרלי לפי מקור

In [ ]:
show('01_exploration/coverage_per_source.png')

### דוגמאות ספקטרומים טהורים מכל מקור

In [ ]:
show('01_exploration/pure_examples_per_source.png')

### השוואה: תערובת סוכר אמיתית מול ספקטרומים טהורים

כאן רואים למה הבעיה קשה — הספקטרום של התערובת הוא שילוב מורכב של הרכיבים, עם חפיפות משמעותיות.

In [ ]:
show('01_exploration/sugar_mixture_vs_pures.png')

---

# שלב 2: יצירת תערובות סינתטיות

## Synthetic Mixture Generation

אין לנו מספיק תערובות אמיתיות מתויגות, לכן יצרנו **גנרטור סינתטי** שמייצר תערובות חדשות **on-the-fly** — כל batch מכיל דוגמאות שמעולם לא נראו.

### תהליך יצירת תערובת:
1. **דגימת K רכיבים** (K בין 1 ל-8) מהמאגר
2. **הגרלת מקדמים** מהתפלגות Dirichlet(α), כאשר α בין 0.3 ל-2.0
3. **סופרפוזיציה ליניארית**: y = Σ cᵢ · rᵢ
4. **הוספת M מסיחים** (distractors): רפרנסים נוספים עם מקדם אמיתי = 0
5. **שחיתויות מציאותיות**:
   - רקע פלואורסצנטי (תמיד חיובי, bump אקספוננציאלי)
   - רעש Gaussian + Poisson (SNR 10-60 dB)
   - הזזת פיקים (±1-3 cm⁻¹)
   - טשטוש (Gaussian broadening)

### תהליך הבנייה צעד אחרי צעד

**צעד 1: ספקטרומים טהורים של הרכיבים**

In [ ]:
show('07_professional/A1_pure_reference_0.png')

**צעד 2: שקילה — כל רכיב מוכפל במקדם הדיריכלה שלו**

In [ ]:
show('07_professional/A2_weighted_components.png')

**צעד 3: סופרפוזיציה — בניית התערובת שלב אחרי שלב**

In [ ]:
show('07_professional/A3_superposition_buildup.png')

**צעד 4: תערובת סופית — עם בייסליין ורעש**

In [ ]:
show('07_professional/A4_final_mixture.png')

### סטטיסטיקות הגנרטור

התפלגות K (מספר רכיבים), גודל מקדמים, ורמות SNR:

In [ ]:
show('02_synth/generator_distributions.png')

### דוגמאות בנייה של 8 תערובות

In [ ]:
show('02_synth/construction_8_samples.png')

### השוואת רמות רעש (SNR)

In [ ]:
show('02_synth/snr_comparison.png')

---

# שלב 3: בייסליין קלאסי — NNLS

## Classical Baseline: Non-Negative Least Squares

לפני אימון מודל למידה עמוקה, הקמנו **בייסליין קלאסי** — שיטה מתמטית ישירה.

### מה זה NNLS?
פותר את בעיית האופטימיזציה:
```
minimize ||y - [R | P] · x||²   subject to x ≥ 0
```
כאשר:
- `y` = ספקטרום התערובת
- `R` = מטריצת רפרנסים
- `P` = בסיס פולינומיאלי (סדר 5) לספיגת הבייסליין
- `x` = מקדמים (תמיד חיוביים)

### יתרונות:
- פשוט, מהיר, אין צורך באימון
- פותר ישירות per-sample — אופטימלי לכל דוגמה

### חסרונות:
- לא לומד דפוסים גלובליים
- לא מתמודד טוב עם רעש כבד או חפיפות ספקטרליות

### דוגמאות פירוק NNLS

In [ ]:
show('03_baselines/nnls_example_0.png')

In [ ]:
show('03_baselines/nnls_example_1.png')

In [ ]:
show('03_baselines/nnls_example_2.png')

### NNLS: מקדמים חזויים מול אמיתיים

In [ ]:
show('03_baselines/nnls_pred_vs_true.png')

### NNLS: MAE לפי מספר רכיבים וSNR

In [ ]:
show('03_baselines/mae_vs_K.png')

In [ ]:
show('03_baselines/mae_vs_snr.png')

---

# שלב 4: בדיקת Overfit

## Overfit Sanity Check

לפני אימון מלא, בדקנו שהארכיטקטורה **מסוגלת ללמוד** על ידי **overfit ל-16 דוגמאות**.

אם המודל לא מצליח לשנן 16 דוגמאות — יש באג בארכיטקטורה או ב-loss.

### תוצאה (v3):
- **Coefficient MAE = 0.012** אחרי 5000 צעדים
- בייסליין חיובי (0% שליליים)
- מקדמי מסיחים < 0.001

### עקומת Loss

In [ ]:
show('04_overfit/loss_curve.png')

### רכיבי ה-Loss

In [ ]:
show('04_overfit/loss_components.png')

### Scatter: מקדמים חזויים מול אמיתיים

In [ ]:
show('04_overfit/coeff_scatter.png')

### דוגמאות פירוק (Overfit)

In [ ]:
show('04_overfit/overfit_example_0.png')

In [ ]:
show('04_overfit/overfit_example_1.png')

---

# שלב 5: אימון מלא (v3)

## Full Training

### קונפיגורציה:
| פרמטר | ערך |
|--------|-----|
| Optimizer | Adam, lr=1e-3, cosine decay ל-1e-5 |
| Batch size | 64 |
| Epochs | 100 (× 500 steps = 50,000 צעדי גרדיינט) |
| Mixed precision | AMP |
| Holdout | 20% מהכימיקלים (64 חומרים) |
| תשתית | Run:AI GPU cluster |

### פונקציית Loss (v3):
```
L = λ_c    · MAE(c_pred, c_true)           דיוק מקדמים        (λ=1.0)
  + λ_r    · MSE(y - b_pred, Σ c·R)        שחזור               (λ=50.0)
  + λ_sad  · SAD(y - b_pred, Σ c·R)        מרחק זווית ספקטרלי  (λ=1.0)
  + λ_b    · MAE(b_pred, b_true)            הערכת בייסליין      (λ=0.5)
  + λ_l1   · ||c_pred||₁                    דלילות              (λ=0.1)
  + λ_bneg · mean(relu(-b_pred))            אי-שליליות בייסליין (λ=10.0)
```

### שינויים מרכזיים ב-v3:
- **softmax → softplus** — כל מקדם עצמאי, מסיחים יכולים לקבל 0 אמיתי
- **λ_r: 1→50** — loss שחזור היה זניח, עכשיו תורם משמעותית
- **λ_l1: 0.01→0.1** — דלילות חזקה יותר ל-softplus
- **Soft non-negativity** — עונש רך על בייסליין שלילי (ReLU הרג גרדיינטים)

### עקומות אימון v3

In [ ]:
show('06_eval_v3/training_curves.png')

### אימונים קודמים (run01, run02) — לפני v3

In [ ]:
show('05_training/run02/learning_curves.png')

In [ ]:
show('05_training/run02/loss_components.png')

---

# שלב 6: הערכה והשוואה — DL v3 מול NNLS

## Evaluation: DL v3 vs NNLS

הערכה על **500 דוגמאות holdout** — כימיקלים שהמודל **מעולם לא ראה** באימון.

### טבלת תוצאות:

| מדד | DL v3 | NNLS | מנצח |
|------|:---:|:---:|:---:|
| MAE (mean) ↓ | 0.122 | **0.093** | NNLS |
| MAE (median) ↓ | 0.089 | **0.063** | NNLS |
| RMSE ↓ | 0.165 | **0.139** | NNLS |
| SAD ↓ | **0.545** | 0.555 | DL v3 |
| R² ↑ | **0.424** | 0.240 | DL v3 |
| AUC-ROC ↑ | **0.759** | 0.737 | DL v3 |
| Spearman ↑ | 0.494 | **0.517** | NNLS |
| DL win rate | **38.8%** | — | — |

### מסקנות:
- **DL v3 מנצח ב-R², AUC-ROC, SAD** — התאמה כללית טובה יותר, זיהוי רכיבים, דמיון צורה
- **NNLS מנצח ב-MAE, RMSE, Spearman** — דיוק מוחלט טוב יותר הודות לאופטימיזציה ישירה per-sample
- DL מנצח ב-38.8% מהדוגמאות — לא מנצח תמיד, אבל יש מקרים שבהם הוא עדיף משמעותית

### Scatter: מקדמים חזויים מול אמיתיים

In [ ]:
show('06_eval_v3/scatter_comparison.png')

### השוואת MAE

In [ ]:
show('06_eval_v3/mae_comparison_bar.png')

### השוואת Spearman

In [ ]:
show('06_eval_v3/spearman_comparison_bar.png')

### ROC Curve — זיהוי רכיבים

עקומת ROC עבור שאלת הזיהוי: "האם רכיב זה נמצא בתערובת?" (סף: c > 0.02)

In [ ]:
show('06_eval_v3/roc_curve.png')

### MAE לפי מספר רכיבים (K)

In [ ]:
show('06_eval_v3/mae_vs_K.png')

### MAE לפי רמת רעש (SNR)

In [ ]:
show('06_eval_v3/mae_vs_snr.png')

### התפלגות שיפור: DL v3 מול NNLS

In [ ]:
show('06_eval_v3/improvement_histogram.png')

### דוגמאות: מקרים שבהם DL v3 מנצח

In [ ]:
show('06_eval_v3/example_dl_wins_0.png')

In [ ]:
show('06_eval_v3/example_dl_wins_1.png')

In [ ]:
show('06_eval_v3/example_dl_wins_2.png')

### דוגמאות: מקרים שבהם NNLS מנצח

In [ ]:
show('06_eval_v3/example_nnls_wins_0.png')

In [ ]:
show('06_eval_v3/example_nnls_wins_1.png')

In [ ]:
show('06_eval_v3/example_nnls_wins_2.png')

---

# שלב 7: גרפים מקצועיים והשוואה מקיפה

## Professional Plots & Comprehensive Comparison

### האתגר: מהספקטרום הנצפה — לפירוק

In [ ]:
show('07_professional/B1_challenge.png')

### פירוק DL v3

In [ ]:
show('07_professional/B2_model_decomposition.png')

### פירוק NNLS

In [ ]:
show('07_professional/B3_nnls_decomposition.png')

### השוואת מקדמים: True vs DL v3 vs NNLS

In [ ]:
show('07_professional/B4_coefficient_comparison.png')

### דשבורד מדדים מקיף: DL v3 מול NNLS

In [ ]:
show('07_professional/D1_metrics_comparison.png', width=1100)

### ROC Curve — DL v3 מול NNLS

In [ ]:
show('07_professional/D2_roc_curve.png', width=600)

### Scatter: חזוי מול אמיתי — DL v3 ו-NNLS זה לצד זה

In [ ]:
show('07_professional/D3_scatter_comparison.png', width=1000)

### עמידות ברעש: MAE לפי SNR

In [ ]:
show('07_professional/D4_mae_vs_snr.png')

### סקאלינג עם מורכבות: MAE לפי מספר רכיבים K

In [ ]:
show('07_professional/D5_mae_vs_K_boxplot.png')

### התפלגות MAE: DL v3 מול NNLS

In [ ]:
show('07_professional/D6_improvement_histogram.png')

### הדגמה: חומרים שלא היו באימון (Unseen Chemicals)

כימיקלים מה-holdout — המודל מעולם לא ראה אותם. כל גרף מראה True, DL v3, ו-NNLS.

In [ ]:
show('07_professional/C_demo_unseen_0.png')

In [ ]:
show('07_professional/C_demo_unseen_1.png')

In [ ]:
show('07_professional/C_demo_unseen_2.png')

In [ ]:
show('07_professional/C_demo_unseen_3.png')

In [ ]:
show('07_professional/C_demo_unseen_4.png')

In [ ]:
show('07_professional/C_demo_unseen_5.png')

---

# שלב 8: טסט על נתונים אמיתיים — תערובות סוכר

## Real Data Test: Sugar Mixtures

**9,600 תערובות סוכר פיזיות** שנמדדו במעבדה.  
5 רכיבים: גלוקוז, פרוקטוז, סוכרוז, מלטוז ומים.

### תוצאות:

| מדד | DL v3 | NNLS |
|------|:---:|:---:|
| Reconstruction RMSE | 0.0073 | **0.0003** |
| סכום מקדמים | 2.56 | **1.01** |
| בייסליין חיובי | 67.9% | **100%** |

### Domain Gap — הפער בין סינתטי לאמיתי:
- המודל נותן מקדמים כמעט **אחידים** (~0.2 לכל רכיב) — לא מצליח להבדיל בין סוכרים דומים
- NNLS עובד הרבה יותר טוב כי הוא פותר ישירות per-sample
- סכום מקדמי DL = 2.56 — אין אילוץ סכום ב-softplus

### זה האתגר המרכזי לעבודה הבאה:
1. Fine-tuning על נתונים אמיתיים
2. שיפור augmentation
3. אילוץ סכום מקדמים
4. גישה היברידית DL+NNLS

### דוגמאות פירוק ומקדמים על תערובות סוכר אמיתיות

In [ ]:
show('08_real_data_test_v3/example_00_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_00_coefficients.png')

In [ ]:
show('08_real_data_test_v3/example_01_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_01_coefficients.png')

In [ ]:
show('08_real_data_test_v3/example_02_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_02_coefficients.png')

In [ ]:
show('08_real_data_test_v3/example_03_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_03_coefficients.png')

In [ ]:
show('08_real_data_test_v3/example_04_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_04_coefficients.png')

In [ ]:
show('08_real_data_test_v3/example_05_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_05_coefficients.png')

In [ ]:
show('08_real_data_test_v3/example_06_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_06_coefficients.png')

In [ ]:
show('08_real_data_test_v3/example_07_decomposition.png')

In [ ]:
show('08_real_data_test_v3/example_07_coefficients.png')

---

# סיכום והמשך

## Summary & Next Steps

### מה הושג:
- מודל DL גנרי לפירוק ספקטרלי — עובד על חומרים שלא ראה באימון
- שיפור משמעותי מ-v1 ל-v3 (Spearman: 0.279 → 0.494, MAE: 0.151 → 0.122)
- תשתית מלאה: גנרטור סינתטי, בייסליין קלאסי, אימון מלא, הערכה מקיפה

### היסטוריית גרסאות:

| גרסה | שינויים מרכזיים | MAE | Spearman |
|-------|-----------------|:---:|:---:|
| v1 | Encoder משותף, פלט ליניארי | — | — |
| v2 | Encoder רק לתערובת, softmax, SAD loss | 0.151 | 0.279 |
| **v3** | **softplus, penalty בייסליין, איזון loss** | **0.122** | **0.494** |

### מה צריך לעשות הלאה:
1. **Fine-tuning על נתונים אמיתיים** — שימוש ב-9,600 תערובות הסוכר המתויגות
2. **שיפור augmentation** — בייסליינים מורכבים, ארטיפקטים מציאותיים
3. **אילוץ סכום מקדמים** — loss נוסף: (Σc - 1)²
4. **גישה היברידית** — DL חוזה בייסליין → NNLS פותר מקדמים
5. **Encoder חזק יותר לרפרנסים** — Conv1D במקום Linear בלבד

---

### קישורים:
- **GitHub:** [orelgrBGU/Raman_Spectroscopy_DL](https://github.com/orelgrBGU/Raman_Spectroscopy_DL)
- **מאמרים מרכזיים:** EGU-Net, Georgiev et al. (PNAS 2024), RamanFormer